# 01 — Generate 1M Synthetic Transactions

This notebook generates a 1,000,000-row synthetic transaction dataset using the provided `transactions.csv` file as the statistical seed. It is designed for Google Colab so the generation can run without depending on local laptop memory or a local Python installation.

The assignment asks for synthetic data that keeps the structure and business characteristics of the source dataset, including realistic distributions, extended transaction activity, seasonality, and representative data-quality issues.

## 1. Import libraries and set configuration

The random seed keeps the output reproducible. If another reviewer runs the notebook with the same input and settings, they should get equivalent results.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import files

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

N_ROWS = 1_000_000
START_DATE = "2024-01-01"
END_DATE = "2026-12-31"
OUTPUT_DIR = Path("data/generated")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = "transactions.csv"
OUTPUT_FILE = OUTPUT_DIR / "synthetic_transactions_1m.csv"

## 2. Upload the provided source file

Run this cell and upload the original `transactions.csv` file from the repo path `data/raw/transactions.csv`.

In [ ]:
uploaded = files.upload()

source = pd.read_csv(RAW_FILE)
print(source.shape)
source.head()

## 3. Profile the original dataset

This step learns the observed distributions from the original data. The generator uses these distributions instead of making every field uniformly random.

In [ ]:
def normalize_category(value):
    if pd.isna(value) or str(value).strip() == "":
        return None
    return str(value).strip().title()

profile = {}

profile["channel_probs"] = source["channel"].dropna().astype(str).value_counts(normalize=True)
profile["product_probs"] = source["product_category"].dropna().astype(str).str.title().value_counts(normalize=True)
profile["status_probs"] = source["status"].dropna().astype(str).str.lower().value_counts(normalize=True)
profile["payment_probs"] = source["payment_method"].dropna().astype(str).value_counts(normalize=True)
profile["quantity_probs"] = source["quantity"].dropna().astype(int).value_counts(normalize=True).sort_index()

for name, probs in profile.items():
    print(f"
{name}")
    display(probs.to_frame("share").head(10))

## 4. Learn amount behavior by product category

Order values often differ by category. For example, electronics transactions are usually larger than books or home goods. To preserve this relationship, the generator estimates amount distributions separately by product category.

In [ ]:
src = source.copy()
src["product_category_clean"] = src["product_category"].map(normalize_category)
src["amount_num"] = pd.to_numeric(src["amount"], errors="coerce")
src_valid_amounts = src[(src["amount_num"] > 0) & src["product_category_clean"].notna()]

amount_params = {}
for cat, group in src_valid_amounts.groupby("product_category_clean"):
    log_amount = np.log(group["amount_num"].clip(lower=1))
    amount_params[cat] = (float(log_amount.mean()), float(log_amount.std(ddof=0) or 0.25))

pd.DataFrame(amount_params, index=["log_mean", "log_std"]).T

## 5. Define seasonality and data-quality issue rates

The synthetic data extends activity across a wider period. It includes slightly higher weekend activity, higher November/December activity, and an April campaign-like bump. It also injects messy records similar to the source file so the downstream cleaning pipeline can be tested.

In [ ]:
days = pd.date_range(START_DATE, END_DATE, freq="D")

weekday_weight = np.where(days.dayofweek >= 5, 1.25, 1.0)
month_weight = np.where(days.month.isin([11, 12]), 1.25, 1.0)
april_campaign_weight = np.where((days.month == 4) & (days.day >= 15) & (days.day <= 25), 1.7, 1.0)

day_weights = weekday_weight * month_weight * april_campaign_weight
day_p = day_weights / day_weights.sum()

issue_rates = {
    "null_timestamp": 0.0015,
    "malformed_timestamp": 0.0015,
    "null_channel": 0.0050,
    "null_product": 0.0035,
    "null_amount": 0.0100,
    "zero_amount": 0.0020,
    "negative_amount": 0.0020,
    "duplicate_rows": 0.0080,
}

issue_rates

## 6. Generate synthetic records

The generation is chunked for performance and memory control. The notebook writes each chunk to CSV instead of keeping every chunk in memory forever.

In [ ]:
def generate_synthetic_transactions(n_rows=N_ROWS, output_path=OUTPUT_FILE, chunk_size=250_000):
    channel_probs = profile["channel_probs"]
    product_probs = profile["product_probs"]
    status_probs = profile["status_probs"]
    payment_probs = profile["payment_probs"]
    quantity_probs = profile["quantity_probs"]

    customers = source["customer_id"].dropna().astype(str).unique()
    max_customer_num = max(
        int(str(c).replace("C", "")) for c in customers if str(c).replace("C", "").isdigit()
    )
    expanded_customer_count = max(max_customer_num * 20, 50_000)

    first_chunk = True
    next_id = 1_000_000

    for offset in range(0, n_rows, chunk_size):
        size = min(chunk_size, n_rows - offset)

        selected_days = rng.choice(days, size=size, p=day_p)
        seconds = rng.integers(0, 24 * 60 * 60, size=size)
        timestamps = selected_days + pd.to_timedelta(seconds, unit="s")

        categories = rng.choice(product_probs.index.values, size=size, p=product_probs.values)
        amounts = np.empty(size)

        for cat in product_probs.index.values:
            mask = categories == cat
            mu, sigma = amount_params.get(cat, (4.0, 0.5))
            amounts[mask] = rng.lognormal(mean=mu, sigma=sigma, size=int(mask.sum()))

        records = pd.DataFrame({
            "transaction_id": [f"SYN{next_id + i}" for i in range(size)],
            "order_timestamp": timestamps.strftime("%Y-%m-%d %H:%M:%S"),
            "channel": rng.choice(channel_probs.index.astype(str).values, size=size, p=channel_probs.values),
            "product_category": categories,
            "quantity": rng.choice(quantity_probs.index.values, size=size, p=quantity_probs.values),
            "amount": np.round(amounts, 2),
            "payment_method": rng.choice(payment_probs.index.astype(str).values, size=size, p=payment_probs.values),
            "customer_id": [f"C{x}" for x in rng.integers(1, expanded_customer_count + 1, size=size)],
            "status": rng.choice(status_probs.index.values, size=size, p=status_probs.values),
        })

        next_id += size

        # Inject representative data-quality issues.
        for col, rate in [
            ("order_timestamp", issue_rates["null_timestamp"]),
            ("channel", issue_rates["null_channel"]),
            ("product_category", issue_rates["null_product"]),
            ("amount", issue_rates["null_amount"]),
        ]:
            mask = rng.random(size) < rate
            records.loc[mask, col] = np.nan

        malformed_mask = rng.random(size) < issue_rates["malformed_timestamp"]
        records.loc[malformed_mask, "order_timestamp"] = "2026-13-01 99:99"

        zero_mask = rng.random(size) < issue_rates["zero_amount"]
        records.loc[zero_mask, "amount"] = 0

        negative_mask = rng.random(size) < issue_rates["negative_amount"]
        records.loc[negative_mask, "amount"] = -np.abs(records.loc[negative_mask, "amount"].astype(float))

        dup_count = int(size * issue_rates["duplicate_rows"])
        if dup_count > 0:
            dup_rows = records.sample(n=dup_count, random_state=RNG_SEED + offset)
            records = pd.concat([records, dup_rows], ignore_index=True)

        records.to_csv(output_path, mode="w" if first_chunk else "a", index=False, header=first_chunk)
        first_chunk = False
        print(f"Wrote chunk ending at base row {offset + size:,}")

    return output_path

# Small smoke test first. This proves the logic works before generating 1M rows.
test_file = OUTPUT_DIR / "synthetic_transactions_test.csv"
generate_synthetic_transactions(n_rows=10_000, output_path=test_file, chunk_size=10_000)

pd.read_csv(test_file).head()

## 7. Generate the full 1M-row dataset

Run this cell after the 10,000-row smoke test succeeds. The final file will have more than 1,000,000 rows because duplicate rows are intentionally added as representative data-quality issues.

In [ ]:
generated_file = generate_synthetic_transactions(
    n_rows=1_000_000,
    output_path=OUTPUT_FILE,
    chunk_size=250_000,
)

print(f"Generated file: {generated_file}")

## 8. Validate the generated output

This confirms that the dataset has the expected structure, date range, messy records, and at least 1M base records.

In [ ]:
synthetic = pd.read_csv(OUTPUT_FILE)

summary = {
    "rows_including_duplicates": len(synthetic),
    "columns": list(synthetic.columns),
    "duplicate_transaction_ids": synthetic["transaction_id"].duplicated().sum(),
    "null_timestamps": synthetic["order_timestamp"].isna().sum(),
    "null_channels": synthetic["channel"].isna().sum(),
    "null_products": synthetic["product_category"].isna().sum(),
    "null_amounts": synthetic["amount"].isna().sum(),
    "zero_amounts": (pd.to_numeric(synthetic["amount"], errors="coerce") == 0).sum(),
    "negative_amounts": (pd.to_numeric(synthetic["amount"], errors="coerce") < 0).sum(),
}

summary

## 9. Compare source and synthetic distributions

These checks help confirm that the generated dataset preserves the original business characteristics rather than creating unrealistic random data.

In [ ]:
def compare_distribution(column, clean_func=lambda x: x):
    source_dist = source[column].dropna().map(clean_func).value_counts(normalize=True).rename("source_share")
    synthetic_dist = synthetic[column].dropna().map(clean_func).value_counts(normalize=True).rename("synthetic_share")
    return pd.concat([source_dist, synthetic_dist], axis=1).fillna(0).sort_values("source_share", ascending=False)

print("Channel distribution")
display(compare_distribution("channel"))

print("Product category distribution")
display(compare_distribution("product_category", lambda x: str(x).title()))

print("Status distribution")
display(compare_distribution("status", lambda x: str(x).lower()))

## 10. Download the generated file if needed

The 1M-row CSV is intentionally not committed to GitHub because it is large and reproducible. The repo should contain the code and notebook, not the heavy generated artifact.

In [ ]:
# Uncomment this line only if you need to download the generated CSV from Colab.
# files.download(str(OUTPUT_FILE))